# Analyses

This notebook contains the business questions and statistical analyses.

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
from sqlalchemy import create_engine

## Loading the dataset

In [ ]:
server = r'LAPTOP-OCKRNGU5\SQLEXPRESS'
database = 'SupplyChainDB'

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

engine = create_engine(connection_string)

query = """
SELECT
  order_id,
  order_customer_id AS customer_id,
  order_date_dateorders AS order_datetime,
  shipping_date_dateorders AS shipping_datetime,
  days_for_shipment_scheduled AS scheduled_ship_days,
  days_for_shipping_real AS actual_ship_days,
  (days_for_shipping_real - days_for_shipment_scheduled) AS shipping_delay_days,
  CASE
    WHEN late_delivery_risk IS NOT NULL THEN late_delivery_risk
    WHEN (days_for_shipping_real - days_for_shipment_scheduled) > 0 THEN 1
    ELSE 0
  END AS is_late_flag,
  delivery_status,
  shipping_mode,
  order_status,
  order_region,
  order_state,
  market,
  order_country,
  order_city,
  product_category_id,
  category_name,
  order_item_product_price,
  order_item_quantity,
  order_item_discount,
  order_item_discount_rate,
  order_item_total,
  sales,
  benefit_per_order,
  order_profit_per_order,
  order_item_profit_ratio,
  CASE
    WHEN sales = 0 OR sales IS NULL THEN NULL
    ELSE ROUND(order_profit_per_order / NULLIF(sales, 0), 6)
  END AS profit_margin,
  DATEPART(year, order_date_dateorders) AS order_year,
  DATEPART(month, order_date_dateorders) AS order_month,
  DATEPART(weekday, order_date_dateorders) AS order_weekday
FROM dbo.supply_chain_data
WHERE
  days_for_shipping_real IS NOT NULL
  AND days_for_shipment_scheduled IS NOT NULL
  AND (order_profit_per_order IS NOT NULL OR sales IS NOT NULL);
"""

supply_chain = pd.read_sql(query, engine)
supply_chain.style

## Does shipping mode significantly affect delivery performance?

A question would rise if the different late delivery rates between shipping modes significantly different.

Initial Hypothesis:

H₀: Shipping mode and late delivery are independent.

H₁: Shipping mode and late delivery are associated.

In [6]:
table = pd.crosstab(
    supply_chain['shipping_mode'],
    supply_chain['is_late_flag']
)

chi2, p, dof, expected = chi2_contingency(table)

print(f"Chi-square: {chi2:.2f}")
print(f"p-value: {p:.4g}")

Chi-square: 37716.04
p-value: 0


With the p-value being 0 and p < 0, then we have evidence that shipping mode and late delivery are associated.

## Does shipping delay affect profitability?

One of the core questions of the project.

In [8]:
supply_chain.groupby('is_late_flag')['benefit_per_order'].agg(
    ['count', 'mean', 'median', 'std']
)

,count,mean,median,std
is_late_flag,,,,
0,81542,22.403808,31.68,103.163253
1,98977,21.621707,31.43,105.467754


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

Q1 = supply_chain['benefit_per_order'].quantile(0.25)
Q3 = supply_chain['benefit_per_order'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

filtered = supply_chain[
    (supply_chain['benefit_per_order'] >= lower_bound) &
    (supply_chain['benefit_per_order'] <= upper_bound)
]

late = filtered.loc[filtered['is_late_flag'] == 1, 'benefit_per_order']
on_time = filtered.loc[filtered['is_late_flag'] == 0, 'benefit_per_order']

t_stat, p_value = ttest_ind(late, on_time, equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)


t-statistic: -1.573466431717265
p-value: 0.11561288626748353
